# README

## Overview

This script facilitates the extraction of place details from Google Maps Place IDs provided in a CSV file. It uses the Google Maps API to retrieve details such as the address, business status, phone number, and website for each place ID.

## Requirements

1. **Python 3.7+**
2. **Pandas Library**
3. **Googlemaps Library**
4. **Ipywidgets Library**
5. **Google Maps API Key**

## Steps to Use the Script

### Step 1: Upload the CSV File

The script starts with a file upload widget to upload your CSV file. The file should contain a column for `Place ID`.

### Step 2: Enter the API Key

Enter your Google Maps API key in the provided text input widget. Ensure the key starts with `AIza`.

### Step 3: Submit the Form

Click the "Submit" button to process the data.

### Step 4: Results

The script will read the uploaded CSV file, extract the `Place ID` column, and use the Google Maps API to retrieve details for each place. The results will be displayed in a dataframe showing the `Place ID`, `address`, `business_status`, `phone`, and `website`. A CSV file named `updated_place_details.csv` will be generated containing the results.

### Step 5: Download the Results

A download button will be provided to download the CSV file with the place details.


In [ ]:
# Step 1: Install necessary packages
!pip install pandas openpyxl googlemaps

In [ ]:
import pandas as pd
import googlemaps
from google.colab import files
import ipywidgets as widgets
from IPython.display import display
import io

# Step 1: Upload the csv file
upload_button = widgets.FileUpload(
    accept='.csv',  # Accept only csv files
    multiple=False  # Accept single file only
)
display(upload_button)

# Step 2: Input for API key
api_key_input = widgets.Text(
    description="API Key:",
    placeholder="Enter your Google Maps API key"
)
display(api_key_input)

# Step 3: Input for lookup limit
lookup_limit_input = widgets.IntText(
    description="Lookup Limit:",
    value=10,
    min=1
)
display(lookup_limit_input)

# Step 4: Submit button to process the data
submit_button = widgets.Button(description="Submit")
display(submit_button)

def handle_submit(button):
    uploaded_file = next(iter(upload_button.value.values()))
    content = uploaded_file['content']
    file_stream = io.BytesIO(content)

    # Step 3: Load the csv file
    df = pd.read_csv(file_stream)

    # Step 5: Extract place names and coordinates
    places = df[['Name', 'Zip']]  # Change these to the correct column names if different

    # Step 6: Set up Google Maps API
    api_key = api_key_input.value  # Get the API key from the input
    if not api_key.startswith("AIza"):
        print("Invalid API key provided.")
        return

    gmaps = googlemaps.Client(key=api_key)

    def get_location_id(place_name, zip_code):
        try:
            # Perform a text search using the place name and ZIP code
            query = f"{place_name} {zip_code}"
            result = gmaps.places(query)
            place_id = result['results'][0]['place_id'] if result['results'] else None
            if place_id is None:
                print(f"Error: No place ID found for {place_name} in {zip_code}")
            return place_id
        except Exception as e:
            print(f"Exception error for {place_name} in {zip_code}: {e}")
            return None

    # Get lookup limit from input
    cost_saving_limit = lookup_limit_input.value

    # Step 8: Get location IDs for the first 'cost_saving_limit' places to save on billing
    limited_places = places.head(cost_saving_limit)
    location_ids = limited_places.apply(lambda x: get_location_id(x['Name'], x['Zip']), axis=1)

    # Combine the original place names with their respective location IDs
    results = pd.DataFrame({'Name': limited_places['Name'], 'Place ID': location_ids})
    print(results)

    # Save the place names and IDs to a CSV file
    output_file_path = 'place_ids_with_names.csv'
    results.to_csv(output_file_path, index=False)

    # Create a download button for the output file
    download_button = widgets.Button(description="Download Results")
    display(download_button)

    def handle_download(button):
        files.download(output_file_path)

    download_button.on_click(handle_download)

# Bind the submit button to the handler
submit_button.on_click(handle_submit)

def handle_submit(button):
    uploaded_file = next(iter(upload_button.value.values()))
    content = uploaded_file['content']
    file_stream = io.BytesIO(content)

    # Step 3: Load the csv file
    df = pd.read_csv(file_stream)

    # Step 5: Extract place names and coordinates
    places = df[['Name', 'Zip']]  # Change these to the correct column names if different

    # Step 6: Set up Google Maps API
    api_key = api_key_input.value  # Get the API key from the input
    if not api_key.startswith("AIza"):
        print("Invalid API key provided.")
        return

    gmaps = googlemaps.Client(key=api_key)

    def get_location_id(place_name, zip_code):
        try:
            # Perform a text search using the place name and ZIP code
            query = f"{place_name} {zip_code}"
            result = gmaps.places(query)
            place_id = result['results'][0]['place_id'] if result['results'] else None
            if place_id is None:
                print(f"Error: No place ID found for {place_name} in {zip_code}")
            return place_id
        except Exception as e:
            print(f"Exception error for {place_name} in {zip_code}: {e}")
            return None

    # Get lookup limit from input
    cost_saving_limit = lookup_limit_input.value

    # Step 8: Get location IDs for the first 'cost_saving_limit' places to save on billing
    limited_places = places.head(cost_saving_limit)
    location_ids = limited_places.apply(lambda x: get_location_id(x['Name'], x['Zip']), axis=1)

    # Combine the original place names with their respective location IDs
    results = pd.DataFrame({'Name': limited_places['Name'], 'Place ID': location_ids})
    print(results)

    # Save the place names and IDs to a CSV file
    output_file_path = 'place_ids_with_names.csv'
    results.to_csv(output_file_path, index=False)

    # Create a download button for the output file
    download_button = widgets.Button(description="Download Results")
    display(download_button)

    def handle_download(button):
        files.download(output_file_path)

    download_button.on_click(handle_download)

# Bind the submit button to the handler
submit_button.on_click(handle_submit)

